# Resultados: figuras y tablas del artículo

Este notebook lee los JSON de resultados (caché de aptitud, registros de feromona de MMAS y, cuando existan, la evaluación final en prueba) y exporta al artículo:

| Archivo | Contenido |
|---|---|
| `paper/figuras/convergencia.pdf` | mejor aptitud frente a número de evaluaciones (MMAS, aleatoria y por defecto) |
| `paper/figuras/feromona.pdf` | evolución de la feromona por hiperparámetro |
| `paper/tablas/configuraciones.tex` | tabla de mejores configuraciones |
| `paper/tablas/resultados.tex` | tabla de resultados en prueba con marcas de significancia |
| `paper/tablas/ejemplos_en-la.tex`, `paper/tablas/ejemplos_en-el.tex` | traducciones de ejemplo |

**Para regenerar todo con los resultados reales basta con cambiar `CARPETA_ENTRADA`** en la celda siguiente (p. ej. a la carpeta fusionada de `buscar.py --salida`). El tipo de aptitud se detecta solo: se usa `real` si existe y, si no, `falsa`.

Los resultados que aún no existen (búsqueda aleatoria, configuración por defecto, evaluación en prueba, griego) no detienen el notebook: se muestra un aviso y las tablas se exportan con «--» o con una marca `\pendiente`.

In [ ]:
# --- Configuración: lo único que hay que cambiar -------------------------
from pathlib import Path

CARPETA_ENTRADA = Path("resultados_ejemplo")  # resultados reales: p. ej. Path("../resultados")
# ---------------------------------------------------------------------------

CARPETA_ARTICULO = Path("../paper")
CARPETA_FIGURAS = CARPETA_ARTICULO / "figuras"
CARPETA_TABLAS = CARPETA_ARTICULO / "tablas"
PAR_BUSQUEDA = "en-la"  # la búsqueda se hace sobre latín (PRD)

In [ ]:
import sys
import warnings

sys.path.insert(0, str(Path("..").resolve()))  # raíz del repositorio
warnings.simplefilter("always")  # mostrar todos los avisos de resultados faltantes

from IPython.display import Markdown, display

from traduccion import busqueda, resultados

## Carga de resultados de la búsqueda

In [ ]:
TIPO_APTITUD = resultados.detectar_tipo_aptitud(CARPETA_ENTRADA, PAR_BUSQUEDA)
if TIPO_APTITUD is None:
    raise FileNotFoundError(f"No hay resultados de búsqueda en {CARPETA_ENTRADA / PAR_BUSQUEDA}")
if TIPO_APTITUD == "falsa":
    warnings.warn("Se usan resultados con la APTITUD FALSA: las figuras y tablas no son resultados reales.")

cache = resultados.cargar_cache(CARPETA_ENTRADA, PAR_BUSQUEDA, TIPO_APTITUD)
registros_mmas = resultados.cargar_registros_feromona(CARPETA_ENTRADA, PAR_BUSQUEDA, TIPO_APTITUD)
print(f"Entrada: {CARPETA_ENTRADA} | par: {PAR_BUSQUEDA} | aptitud: {TIPO_APTITUD}")
print(f"Evaluaciones en la caché: {len(cache)} | semillas de MMAS: {sorted(registros_mmas)}")

## Curva de mejor aptitud frente a número de evaluaciones

Con varias semillas se dibuja la media y una banda entre el mínimo y el máximo. La configuración por defecto es una línea horizontal (una sola evaluación).

In [ ]:
curvas = resultados.curvas_convergencia(cache, registros_mmas)
for metodo in resultados.METODOS_BUSQUEDA:
    if metodo not in curvas:
        warnings.warn(f"No hay evaluaciones del método «{metodo}»; no aparece en la curva.")

registro_defecto = resultados.registro_defecto(cache)
if registro_defecto is None and TIPO_APTITUD == "falsa":
    # Con la aptitud falsa el valor de la configuración por defecto se conoce sin entrenar.
    registro_defecto = {"aptitud": busqueda.aptitud_falsa(resultados.CONFIGURACION_DEFECTO)}
if registro_defecto is None:
    warnings.warn("No hay evaluación de la configuración por defecto; no aparece en la curva.")
aptitud_defecto = registro_defecto["aptitud"] if registro_defecto else None

etiqueta_y = "Mejor aptitud (chrF en validación)" if TIPO_APTITUD == "real" else "Mejor aptitud (falsa)"
figura = resultados.figura_convergencia(curvas, aptitud_defecto, etiqueta_y)
resultados.guardar_figura(figura, CARPETA_FIGURAS / "convergencia.pdf")
print("Exportada:", CARPETA_FIGURAS / "convergencia.pdf")

## Evolución de la feromona

Se dibuja la semilla de MMAS que encontró la mejor configuración. El color es la probabilidad de elegir cada valor (feromona normalizada por hiperparámetro, ∝ τ^α).

In [ ]:
if registros_mmas:
    semilla_feromona = max(registros_mmas, key=lambda s: registros_mmas[s]["mejor"]["aptitud"])
    figura = resultados.figura_feromona(registros_mmas[semilla_feromona])
    resultados.guardar_figura(figura, CARPETA_FIGURAS / "feromona.pdf")
    print(f"Semilla {semilla_feromona}. Exportada:", CARPETA_FIGURAS / "feromona.pdf")
else:
    warnings.warn("No hay registros de feromona: no se exporta feromona.pdf.")

## Tabla de mejores configuraciones

In [ ]:
mejores = resultados.mejor_por_metodo(cache, registros_mmas)
for metodo in resultados.METODOS_BUSQUEDA:
    if metodo not in mejores:
        warnings.warn(f"No hay mejor configuración del método «{metodo}»; su columna queda con «--».")
tabla = resultados.tabla_configuraciones(registro_defecto, mejores)
resultados.escribir_texto(CARPETA_TABLAS / "configuraciones.tex", tabla)
print(tabla)

## Tabla de resultados en prueba

$^\dagger$ marca las configuraciones cuya diferencia con MMAS del mismo par es significativa ($p < 0.05$, bootstrap pareado). Requiere `prueba/resultados_prueba.json` (issue 009).

In [ ]:
resultados_prueba = resultados.cargar_resultados_prueba(CARPETA_ENTRADA)
tabla = resultados.tabla_resultados_prueba(resultados_prueba)
resultados.escribir_texto(CARPETA_TABLAS / "resultados.tex", tabla)
print(tabla)

## Traducciones de ejemplo

In [ ]:
for par in ("en-la", "en-el"):
    ejemplos = resultados.cargar_ejemplos(CARPETA_ENTRADA, par)
    fragmento = resultados.tabla_ejemplos(ejemplos, par)
    resultados.escribir_texto(CARPETA_TABLAS / f"ejemplos_{par}.tex", fragmento)
    print(f"{par}: {len(ejemplos)} ejemplos ->", CARPETA_TABLAS / f"ejemplos_{par}.tex")

## Uso en el artículo

Las figuras las incluye `paper/main.tex` mediante `\figuraopcional{figuras/convergencia.pdf}` y `\figuraopcional{figuras/feromona.pdf}`: si el PDF existe sustituye al recuadro de marcador.

Las tablas son fragmentos `tabular` sin `table`, de modo que el artículo conserva su título y su etiqueta; dentro del entorno `table` correspondiente:

```latex
\input{tablas/configuraciones.tex}   % tab:configuraciones
\input{tablas/resultados.tex}        % tab:resultados
\input{tablas/ejemplos_en-la.tex}    % análisis cualitativo
```